In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler



In [42]:
# Cargar el conjunto de datos
df = pd.read_csv('demanda_productos.csv')


In [44]:
df.info()
print(df)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ProductoID        1000 non-null   int64  
 1   Categoria         1000 non-null   object 
 2   Precio            1000 non-null   float64
 3   Estacion          1000 non-null   object 
 4   Promocion         1000 non-null   int64  
 5   VentasHistoricas  1000 non-null   int64  
 6   Demanda           1000 non-null   int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 54.8+ KB
     ProductoID   Categoria   Precio   Estacion  Promocion  VentasHistoricas  \
0            52  Accesorios  42065.0      Otoño          0               882   
1            93     Calzado  50453.0     Verano          0               472   
2            15        Ropa  58548.0     Verano          0               152   
3            72     Calzado  50704.0  Primavera          1               843   
4        

In [ ]:
# Convertir las variables categóricas en variables dummy
df = pd.get_dummies(df, columns=['Categoria', 'Estacion'], drop_first=True)

In [32]:
# Definir las características (X) y la variable objetivo (y)
X = df.drop(['ProductoID', 'Demanda'], axis=1)
y = df['Demanda']


In [33]:
print(X.shape, y.shape)

(1000, 8) (1000,)


In [34]:
print(X.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Precio              1000 non-null   float64
 1   Promocion           1000 non-null   int64  
 2   VentasHistoricas    1000 non-null   int64  
 3   Categoria_Calzado   1000 non-null   bool   
 4   Categoria_Ropa      1000 non-null   bool   
 5   Estacion_Otoño      1000 non-null   bool   
 6   Estacion_Primavera  1000 non-null   bool   
 7   Estacion_Verano     1000 non-null   bool   
dtypes: bool(5), float64(1), int64(2)
memory usage: 28.4 KB
None


In [35]:
# Normalizar las características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)



In [37]:
print(X_scaled)

[[ 0.13113766 -0.93980805 -0.12079067 ...  1.71830416 -0.60045334
  -0.5557882 ]
 [ 0.71442237 -0.93980805  0.6671761  ... -0.58196914 -0.60045334
   1.79924655]
 [ 1.27739185 -0.93980805  0.79077873 ... -0.58196914 -0.60045334
   1.79924655]
 ...
 [ 1.03964889 -0.93980805  0.45859666 ... -0.58196914  1.66540833
  -0.5557882 ]
 [ 0.23802134 -0.93980805 -0.38344626 ... -0.58196914 -0.60045334
   1.79924655]
 [ 0.03157479  1.06404707 -0.22121781 ... -0.58196914  1.66540833
  -0.5557882 ]]


In [ ]:
# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [17]:
from sklearn.tree import DecisionTreeRegressor


# Crear y entrenar el modelo de árbol de decisión
model_tree = DecisionTreeRegressor(random_state=42)
model_tree.fit(X_train, y_train)

# Hacer predicciones sobre el conjunto de prueba
y_pred_tree = model_tree.predict(X_test)


In [21]:
from sklearn.model_selection import GridSearchCV

from sklearn.model_selection import GridSearchCV

# Definir los hiperparámetros a ajustar
param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Crear el modelo de árbol de decisión
tree = DecisionTreeRegressor(random_state=42)

# Realizar la búsqueda de cuadrícula
grid_search = GridSearchCV(estimator=tree, param_grid=param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Mejor modelo de árbol de decisión
best_tree = grid_search.best_estimator_

# Hacer predicciones sobre el conjunto de prueba
y_pred_best_tree = best_tree.predict(X_test)

# Evaluar el modelo
mse_best_tree = mean_squared_error(y_test, y_pred_best_tree)
print(f"Error cuadrático medio (MSE) con Árbol de Decisión ajustado: {mse_best_tree}")

# Guardar las predicciones en un archivo CSV
predicciones_best_tree = pd.DataFrame({'Demanda_Predicha': y_pred_best_tree})
predicciones_best_tree.to_csv('predicciones_demanda_best_tree.csv', index=False)

print("Predicciones guardadas en 'predicciones_demanda_best_tree.csv'")


Error cuadrático medio (MSE) con Árbol de Decisión ajustado: 28131.51901683696
Predicciones guardadas en 'predicciones_demanda_best_tree.csv'


In [22]:
# Crear y entrenar el modelo de regresión lineal
model = LinearRegression()
model.fit(X_train, y_train)

# Hacer predicciones sobre el conjunto de prueba
y_pred = model.predict(X_test)

In [23]:

# Evaluar el modelo
mse = mean_squared_error(y_test, y_pred_tree)
print(f"Error cuadrático medio (MSE): {mse}")



Error cuadrático medio (MSE): 60579.075


In [13]:
# Guardar las predicciones en un archivo CSV
predicciones = pd.DataFrame({'Demanda_Predicha': y_pred})
predicciones.to_csv('predicciones_demanda.csv', index=False)

print("Predicciones guardadas en 'predicciones_demanda.csv'")

Predicciones guardadas en 'predicciones_demanda.csv'
